In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
#load dataset
df=pd.read_csv(r"C:\Users\anshi\OneDrive\Desktop\credit risk prediction\dataset\loan_risk_prediction_dataset.csv")
df.head()

In [ ]:
#explore dataset
print(df.shape)
print(df.columns)
print(df.info())
print(df.describe())


In [ ]:
#check for missing values
print(df.isnull().sum())

In [ ]:
#Data cleaning 
#1.Handle missing values
for col in ['Income','CreditScore']:
    df[col]=df[col].fillna(df[col].mean())
df['Education']=df['Education'].fillna(df['Education'].mode()[0])


In [ ]:
#checking for null value again
df.isnull().sum()


In [ ]:
#checking duplicates
duplicate=df[df.duplicated()]
print(duplicate)

In [ ]:
#EXPLORATORY DATA ANALYSIS
target_count=df['LoanApproved'].value_counts()
print(target_count)

sns.countplot(x='LoanApproved', data=df)
plt.savefig(
    "images/target_count.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

##observation--we have 3849 rejected loan apllication and 1151 approved loan apllication  (dataset is imbalanced)

In [ ]:
#distribution analysis
features=['Income','LoanAmount','CreditScore']
colour_hist=['skyblue','green','violet']
colour_box=['salmon','lightgreen','cyan']
fig,axes=plt.subplots(2,3,figsize=(18,10))

for i,col in enumerate(features):
    #histogram
    sns.histplot(df[col],kde=True,color=colour_hist[i],ax=axes[0,i])
    axes[0,i].set_title(f"{col} distribution")
    #boxplot
    sns.boxplot(x=df[col],ax=axes[1,i],color=colour_box[i])
    axes[1,i].set_title(f"{col} boxplot")
plt.tight_layout()
plt.savefig(
    "images/distribution.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()




###INSIGHT-BEFORE CLIPPING
##Income
Income approximately follows a normal distribution with a peak around 50k and a range extending slightly into negative values due to anomalies.

##LoanAmount
LoanAmount shows slight positive skewness with a peak near 20k and contains some invalid negative values.

##CreditScore
CreditScore is distributed across the range of 300–850 with no major outliers or sharp peaks.

##Overall
Initial EDA revealed invalid negative values, indicating the need for preprocessing. -->

#PREPROCESSING
CONVERTED ALL NEGATIVE VALUE INTO ZERO USING CLIPPING


In [ ]:
# AS LOAN AMOUNT AND INCOME HAVE NEGATIVE VALUE , WE WILL CLIP IT TO ZERO  BECAUSE IN REAL WORLD NEGATIVE VALUE IS IMPOSSIBLE
print((df['Income'] < 0).sum())
print((df['LoanAmount'] < 0).sum())
df['Income'] = df['Income'].clip(lower=0)
df['LoanAmount'] = df['LoanAmount'].clip(lower=0)

#visualization after clipping negative values
features=['Income','LoanAmount','CreditScore']
colour_hist=['skyblue','green','violet']
colour_box=['salmon','lightgreen','cyan']
fig,axes=plt.subplots(2,3,figsize=(18,10))

for i,col in enumerate(features):
    #histogram
    sns.histplot(df[col],kde=True,color=colour_hist[i],ax=axes[0,i])
    axes[0,i].set_title(f"{col} distribution")
    #boxplot
    sns.boxplot(x=df[col],ax=axes[1,i],color=colour_box[i])
    axes[1,i].set_title(f"{col} boxplot")
plt.tight_layout()
plt.savefig(
    "images/distribution_aft_clipping.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

###INSIGHT
#After Clipping

#Income
After clipping, Income became more domain-consistent while retaining its peak around 50k and approximately normal distribution.

#LoanAmount
LoanAmount distribution became cleaner after removing negative values, with most loans concentrated around 20k.

#CreditScore
CreditScore distribution remained unchanged across the 300–850 range after preprocessing.

#Overall
Clipping improved data quality while preserving the overall feature distributions and peaks.

In [ ]:
#5) bivariate analysis
features=['Income','LoanAmount','CreditScore','YearsExperience']
colors=['skyblue','green','violet','salmon']
fig,axes=plt.subplots(2,2,figsize=(14,10))
axes=axes.flatten()
for i,col in enumerate(features):
    sns.boxplot(x='LoanApproved',y=col,data=df,ax=axes[i],color=colors[i])
    axes[i].set_title(f'{col} vs LoanApproved')
    
plt.tight_layout()
plt.savefig(
    "images/numeric_bivariate_analysis.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()




###INSIGHT
CreditScore appears to have the strongest relationship with loan approval, while Income has moderate influence.
LoanAmount and YearsExperience show comparatively weaker separation between approved and rejected applicants.
However, their final predictive importance should be evaluated during model training.

In [ ]:
#categorical analysis
features=['Education','EmploymentType']
fig,axes=plt.subplots(1,2,figsize=(14,6))
axes=axes.flatten()
for i,col in enumerate(features):
    sns.countplot(x=col,hue="LoanApproved",data=df,ax=axes[i]
               )
    axes[i].set_title(f"{col} vs LoanApproved")
    for container in axes[i].containers:
        axes[i].bar_label(container)
        
plt.tight_layout()
plt.savefig(
    "images/categorical_bivariate_analysis.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

print(pd.crosstab(
    df['Education'],
    df['LoanApproved'],
    normalize='index'
)*100)

print(pd.crosstab(
    df['EmploymentType'],
    df['LoanApproved'],
    normalize='index'
)*100)



### Insight: Education vs Loan Approval

- Loan approvals were observed across all education levels, indicating that education alone was not a decisive factor in loan approval.
- Applicants with Bachelor's and PhD degrees showed slightly higher approval counts compared to other education categories.
- The majority of applicants in every education category were still rejected, suggesting that financial factors such as income, credit score, and loan amount likely had a stronger influence on approval decisions.

### Insight: Employment Type vs Loan Approval

- Employment type showed a stronger relationship with loan approval than education level.
- Salaried and self-employed applicants received significantly more approvals compared to unemployed applicants.
- Unemployed applicants had the lowest approval count, indicating that stable employment status positively influences loan approval decisions.
- This suggests that employment type is an important feature for assessing an applicant's repayment capability and creditworthiness.

In [ ]:
#correlaton heatmap
corr=df.corr(numeric_only=True)
sns.heatmap(corr,annot=True,cmap='viridis')
plt.savefig(
    "images/heatmap.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

### Insight

- Credit Score shows the strongest positive correlation with LoanApproved (0.46), indicating that applicants with higher credit scores are more likely to receive loan approval.
- Income has a weak positive correlation with LoanApproved (0.19), suggesting that higher income slightly increases the likelihood of approval.
- Age, Loan Amount, and Years of Experience have correlations close to zero with LoanApproved, indicating a very weak linear relationship with loan approval decisions.
- Most independent variables show low correlations with each other, suggesting the absence of significant multicollinearity in the dataset.
- Since no pair of features exhibits a very high correlation, all features can be retained for model training without major concerns about redundancy.

In [ ]:
#converting categorical columns into numerical
#ordinal mapping on Education
df['Education'] = df['Education'].str.strip()
df['Gender'] = df['Gender'].str.strip()
df['Education']=df['Education'].map({'High School':1,'Bachelors':2,'Masters':3,'PhD':4})
df['Gender']=df['Gender'].map({'Female':1,'Male':0})
#one hot encoding on city and employement type
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(drop='first',sparse_output=False)
col=['City','EmploymentType']
encoded=encoder.fit_transform(df[col])
encoded_df=pd.DataFrame(encoded,columns=encoder.get_feature_names_out(col),index=df.index)

#merge encoded_df and df
df=pd.concat([df,encoded_df],axis=1)


In [ ]:
#dropping categorical column
df.drop(col,axis=1,inplace=True)

In [ ]:
print(df.head())
print(df.tail())

In [ ]:
#SEPARATE X AND y
X=df.drop('LoanApproved',axis=1)
y=df['LoanApproved']

In [ ]:
#TRAIN-TEST SPLIT
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [ ]:
# APPLY SMOTE TO HANDLE IMBALANCE DATA
from imblearn.over_sampling import SMOTE
smote=SMOTE(random_state=42)
X_train_smote,y_train_smote = smote.fit_resample(X_train,y_train)

In [ ]:
#scaling
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_smote=scaler.fit_transform(X_train_smote)
X_test=scaler.transform(X_test)


In [ ]:
#IMPLEMENTING PERCEPTRON FROM SCRATCH
class perceptron:
    def __init__(self,eta=0.01,n_iter=50,random_state=1):
        self.eta=eta
        self.n_iter=n_iter
        self.random_state=random_state
    def fit(self,X,y):
        rgen=np.random.RandomState(self.random_state)
        self.w=rgen.normal(loc=0.0,scale=0.01,size=X.shape[1])
        self.b=np.float64(0.)
        self.errors_=[]
        for _ in range(self.n_iter):
            errors=0
            for xi,target in zip(X,y):
                update=self.eta * (target-self.predict(xi))
                self.w+=update*xi
                self.b+=update
                errors+=int(update!=0.0)
            self.errors_.append(errors)
        return self
    def net_input(self,X):
        return np.dot(X,self.w)+self.b
    def predict(self,X):
        return np.where(self.net_input(X)>=0.0,1,0)
    
            
# train perceptron model
ppn=perceptron(eta=0.01,n_iter=100)
ppn.fit(X_train_smote,y_train_smote)
print(ppn.w)
print(ppn.b)
y_pred=ppn.predict(X_test)

# model performance
from sklearn.metrics import accuracy_score
print("ACCURACY BY PERCEPTRON MODEL = " , accuracy_score(y_test,y_pred))
from sklearn.metrics import classification_report,confusion_matrix
print("CLASSIFICATION REPORT OF PERCEPTRON MODEL= " , classification_report(y_test,y_pred))
print("CONFUSION MATRIX OF PERCEPTRON MODEL = " , confusion_matrix(y_test,y_pred))

# learning curve
plt.plot(
    range(1, len(ppn.errors_) + 1),
    ppn.errors_,
    marker='o'
)
plt.xlabel('Epochs')
plt.ylabel('Number of updates')
plt.title('Perceptron Learning')
plt.savefig(
    "images/perceptron_learning.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.show()

      

    

# Interpretation

- Perceptron achieved an accuracy of 27.2%, making it the weakest-performing model in this study.
- The model showed a very high recall (1.00) for approved loans but incorrectly classified a large number of rejected loans as approved.
- The fluctuating error curve indicated that the model struggled to learn a stable decision boundary.
- This suggests that the dataset is not perfectly linearly separable, which limits the effectiveness of the Perceptron algorithm.

In [ ]:
#IMPLEMENTING ADALINE 
class adaline:
    def __init__(self,eta=0.01,n_iter=10,shuffle=True,random_state=None):
        self.eta=eta
        self.n_iter=n_iter
        self.shuffle=shuffle
        self.w_initialized=False
        self.random_state=random_state
    def fit(self,X,y):
        self._initialize_weights(X.shape[1])
        self.losses_=[]
        for i in range(self.n_iter):
            if self.shuffle:
                X,y=self._shuffle(X,y)
                losses=[]
                for xi,target in zip(X,y):
                    losses.append(self._update_weights(xi,target))
                avg_loss=np.mean(losses)
                self.losses_.append(avg_loss)
        return self
    def partial_fit(self,X,y):
        if not self.w_initialized:
            self._initialize_weights(X.shape[1])
        if y.ravel().shape[0]>1:
            for xi,target in zip(X,y):
                self._update_weights(xi,target)
        else:
            self._update_weights(X,y)
        return self
    def _shuffle(self,X,y):
        r=self.rgen.permutation(len(y))
        return X[r],y[r]
    def _initialize_weights(self,m):
        self.rgen=np.random.RandomState(self.random_state)
        self.w_=self.rgen.normal(loc=0.0,scale=0.01,size=m)
        self.b_=np.float64(0.)
        self.w_initialized=True
    def _update_weights(self,xi,target):
        output=self.activation(self.net_input(xi))
        error=(target-output)
        self.w_+=self.eta*2.0*xi*(error)
        self.b_+=self.eta*2.0*error
        loss=error**2
        return loss
    def net_input(self,X):
        return np.dot(X,self.w_)+self.b_
    def activation(self,X):
        return X
    def predict(self,X):
        return np.where(self.activation(self.net_input(X))>=0.5,1,0)
        
# train adaline model
ada=adaline(eta=0.0001,n_iter=25,random_state=1)
ada.fit(X_train_smote,y_train_smote)
print(ada.w_)
print(ada.b_)
y_pred=ada.predict(X_test)

# model performance
from sklearn.metrics import accuracy_score
print("ACCURACY BY ADALINE MODEL = " , accuracy_score(y_test,y_pred))
from sklearn.metrics import classification_report,confusion_matrix
print("CLASSIFICATION REPORT OF ADALINE MODEL= " , classification_report(y_test,y_pred))
print("CONFUSION MATRIX OF ADALINE MODEL = " , confusion_matrix(y_test,y_pred))

# learning curve
plt.plot(
    range(1, len(ada.losses_) + 1),
    ada.losses_,
    marker='o'
)
plt.xlabel('Epochs')
plt.ylabel('Average loss')
plt.title('Adaline SGD learning')
plt.grid(True)
plt.savefig(
    "images/adaline_learning.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

        
    

# Interpretation

- Adaline improved the accuracy to 58.0%, significantly outperforming the Perceptron model.
- The loss curve decreased and stabilized after several epochs, indicating successful convergence.
- Gradient descent optimization enabled Adaline to learn more effectively from the training data.
- Despite the improvement, the model still showed limited predictive performance compared to advanced classifiers.

In [ ]:
#USING LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression(class_weight='balanced', max_iter=1000)
lr_model.fit(X_train_smote,y_train_smote)
y_pred=lr_model.predict(X_test)

# model performance
from sklearn.metrics import accuracy_score
print("ACCURACY OF LOGISTIC MODEL = " , accuracy_score(y_test,y_pred))
from sklearn.metrics import classification_report,confusion_matrix
print("CLASSIFICATION REPORT OF LOGISTIC MODEL= " , classification_report(y_test,y_pred))
print("CONFUSION MATRIX OF LOGISTIC MODEL = " , confusion_matrix(y_test,y_pred))

#PERFORMING HYPERPARAMTER TUNING
from sklearn.model_selection import GridSearchCV
params={ 'C': [0.001,0.01,0.1,1,10,100],
    'penalty': ['l1','l2'],
    'solver': ['liblinear']}
grid_lr=GridSearchCV(LogisticRegression(
        class_weight='balanced',
        max_iter=1000
    ),
    param_grid=params,
    cv=5,
    scoring='f1',
    n_jobs=-1)
grid_lr.fit(X_train_smote, y_train_smote)

print("Best Parameters:", grid_lr.best_params_)
print("Best CV Score:", grid_lr.best_score_)

best_lr = grid_lr.best_estimator_

y_pred_lr = best_lr.predict(X_test)
print("ACCURACY OF LOGISTIC MODEL AFTER HYPERPARAMETER TUNING = " , accuracy_score(y_test,y_pred_lr))
print("CLASSIFICATION REPORT OF LOGISTIC MODEL AFTER HYPERPARAMETER TUNING = " , classification_report(y_test,y_pred_lr))
print("CONFUSION MATRIX OF LOGISTIC MODEL AFTER HYPERPARAMETER TUNING = " , confusion_matrix(y_test,y_pred_lr))



#Interpretation-before tuning

- Logistic Regression achieved an accuracy of 85.1%, representing a substantial improvement over Perceptron and Adaline.
- The model achieved a recall of 89% for approved loans, successfully identifying most eligible applicants.
- Hyperparameter tuning produced similar results, indicating that the baseline model was already close to optimal.
- The model demonstrated strong generalization and balanced classification performance.

#Interpretation-after tuning

- GridSearchCV identified C = 10 as the optimal regularization parameter.
- However, the tuned model produced the same test-set performance as the baseline model.
- This indicates that Logistic Regression was already performing near its optimal capacity on this dataset.
- Therefore, the baseline Logistic Regression model was retained for comparison.

In [ ]:
#USING SUPPORT VECTOR MACHINE
from sklearn.svm import SVC

svm = SVC(kernel='linear',class_weight='balanced', random_state=42,probability=True)
svm.fit(X_train_smote, y_train_smote)
y_pred_svm = svm.predict(X_test)
print("ACCURACY OF SVM MODEL = " , accuracy_score(y_test,y_pred_svm))
print("CLASSIFICATION REPORT OF SVM MODEL = " , classification_report(y_test,y_pred_svm))
print("CONFUSION MATRIX OF SVM MODEL = " , confusion_matrix(y_test,y_pred_svm))

#Interpretation

- SVM achieved the highest accuracy of 85.7% among all evaluated models.
- The model achieved the best balance between precision, recall, and F1-score.
- By maximizing the margin between classes, SVM was able to create a more effective decision boundary.
- These results indicate that SVM is the most suitable model for this loan approval prediction task.
- Although SVM achieved the highest accuracy (85.7%), Logistic Regression achieved a slightly higher recall (89%) for approved loans. This indicates that Logistic Regression was marginally better at identifying eligible applicants, whereas SVM was more conservative and reduced false approvals. The performance difference between the two models was relatively small.

#Overall Model Comparison

| Model | Accuracy |
|---------|---------|
| Perceptron | 27.2% |
| Adaline | 58.0% |
| Logistic Regression | 85.1% |
| SVM | 85.7% |

#Observation

- Perceptron achieved the lowest performance because the dataset was not perfectly linearly separable.
- Adaline improved performance through gradient descent optimization and achieved better convergence.
- Logistic Regression significantly outperformed both Perceptron and Adaline, achieving strong classification performance and high recall for approved loans.
- SVM achieved the highest accuracy and reduced false approvals compared to Logistic Regression.
- The performance difference between Logistic Regression and SVM was relatively small, indicating that both models were highly effective for this classification task.

In [ ]:
from sklearn.metrics import roc_curve, auc

# Logistic Regression Probabilities
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

# SVM Probabilities
y_prob_svm = svm.predict_proba(X_test)[:, 1]

# ROC Points
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)

# AUC Scores
auc_lr = auc(fpr_lr, tpr_lr)
auc_svm = auc(fpr_svm, tpr_svm)

print("Logistic Regression AUC =", round(auc_lr, 3))
print("SVM AUC =", round(auc_svm, 3))

# Plot
plt.figure(figsize=(8,6))

plt.plot(
    fpr_lr,
    tpr_lr,
    linewidth=2,
    label=f'Logistic Regression (AUC = {auc_lr:.3f})'
)

plt.plot(
    fpr_svm,
    tpr_svm,
    linewidth=2,
    label=f'SVM (AUC = {auc_svm:.3f})'
)

# Random Classifier
plt.plot(
    [0,1],
    [0,1],
    linestyle='--',
    label='Random Classifier'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.savefig(
    "images/roc-auc.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

## ROC-AUC Analysis

- Logistic Regression achieved an AUC score of 0.907, while SVM achieved a slightly higher AUC score of 0.909.
- Both models achieved AUC values greater than 0.90, indicating excellent classification performance.
- The ROC curves are very close to each other, suggesting that both models have similar discriminative ability.
- SVM slightly outperformed Logistic Regression in terms of overall class separation capability.
- Both models performed significantly better than the random classifier (diagonal line), confirming that the learned patterns are meaningful.


#Conclusion

This project developed a machine learning pipeline for loan approval prediction using demographic, financial, and employment-related features.

The workflow included data cleaning, exploratory data analysis, categorical encoding, feature scaling, and class balancing using SMOTE. Multiple classification algorithms were evaluated, including Perceptron, Adaline, Logistic Regression, and Support Vector Machine.

Perceptron and Adaline were implemented from scratch to understand the fundamentals of linear classification and gradient-based learning. Logistic Regression and SVM demonstrated substantially better predictive performance.

Although SVM achieved the highest accuracy of 85.7%, Logistic Regression achieved a slightly higher recall (89%) for approved loans, indicating better identification of eligible applicants. The difference between the two models was small, suggesting that both models are highly suitable for loan approval prediction.

Overall, the project highlights the importance of data preprocessing, handling class imbalance, and selecting appropriate machine learning algorithms for financial decision-making problems.

In [ ]:
#saving best model-svm
import joblib

joblib.dump(
    svm,
    "loan_approval_model.pkl"
)

print("Model Saved Successfully")

In [ ]:
joblib.dump(
    scaler,
    "scaler.pkl"
)

print("Scaler Saved Successfully")